# Visualizing MammAlps predictions and ground truth

This notebook renders bounding boxes and attributes (species, action, activity,
demographics) on top of a MammAlps video, either from the **ground-truth
annotations** shipped with the
[amathislab/Prompting-MammAlps](https://huggingface.co/datasets/amathislab/Prompting-MammAlps)
dataset, or from a **SALMA prediction JSON** you produced yourself — both use
the same per-video annotation schema, so the same rendering code works for
either.

## 1. Download a sample from the dataset

The full dataset is ~79GB (videos + annotations), so for this tutorial we only
download one camera's worth of videos/annotations plus the small metadata
files, using `allow_patterns`. Adjust `SITE_CAMERA` or drop `allow_patterns`
entirely to fetch more/all of the data.

In [ ]:
import os
import json
from pathlib import Path

import cv2
import ffmpeg
import numpy as np
import supervision as sv
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

REPO_ID = "amathislab/Prompting-MammAlps"
DATA_ROOT = Path("./data")
SITE_CAMERA = "S1/C6"  # one camera's worth of data, enough for this demo

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATA_ROOT,
    allow_patterns=[
        "metadata/label_mapping.json",
        f"videos/test/{SITE_CAMERA}/*",
        f"annotations/test/{SITE_CAMERA}/*",
    ],
)

with open(DATA_ROOT / "metadata" / "label_mapping.json", "r") as f:
    label_mapping = json.load(f)

## 2. Helper functions

`process_video` reads a video and its per-frame detections JSON
(`{"frames": [{"detections": [{"bbox", "track_id", "conf", "attributes": {...}}, ...]}, ...]}`)
and writes an annotated copy. `color_by` controls whether boxes are colored by
track identity, action, or activity.

In [ ]:
def from_detection_to_sv(frame_detection_results, color_by) -> sv.Detections:
    if frame_detection_results["detections"]:
        detections_list = frame_detection_results.get("detections", [])
        xyxy_coord = np.array([d["bbox"] for d in detections_list], dtype=int)

        confidence = np.array([float(d["conf"]) for d in detections_list])
        tracker_id = np.array([int(d["track_id"]) for d in detections_list])

        extra_data = {}
        for attribute_key in ["Species", "Deer_age", "Deer_adult_sex", "Activity", "Action", "Action2"]:
            values = [d["attributes"].get(attribute_key) if "attributes" in d.keys() else None for d in detections_list]
            extra_data[attribute_key] = np.array(values)

        if color_by == "action":
            class_id = np.array([int(label_mapping["actions"][d["attributes"]["Action"]]) if "attributes" in d.keys() else 999 for d in detections_list])
        elif color_by == "activity":
            class_id = np.array([int(label_mapping["activities"][d["attributes"]["Activity"]]) if "attributes" in d.keys() else 999 for d in detections_list])
        elif color_by == "track":
            class_id = np.array([int(d["track_id"]) if "track_id" in d.keys() else 999 for d in detections_list])

        detections = sv.Detections(
            xyxy=xyxy_coord,
            confidence=confidence,
            class_id=class_id,
            tracker_id=tracker_id,
            data=extra_data,
        )
    else:
        detections = sv.Detections.empty()

    return detections


def process_video(
        source_video_path: str,
        detections_path: str,
        target_video_path: str,
        color_by: str,
        show_attributes=True,
) -> None:
    color_lookup = sv.ColorLookup("class")

    box_annotator = sv.BoxAnnotator(color_lookup=color_lookup, thickness=2)
    box_fill_annotator = sv.ColorAnnotator(color_lookup=color_lookup, opacity=.2)
    label_annotator = sv.LabelAnnotator(text_scale=1.8, color_lookup=color_lookup, text_thickness=3, smart_position=False)
    frame_generator = sv.get_video_frames_generator(source_path=source_video_path, stride=1)
    video_info = sv.VideoInfo.from_video_path(video_path=source_video_path)

    with open(detections_path, "r") as f:
        detection_results = json.load(f)["frames"]

    with sv.VideoSink(target_path=target_video_path, video_info=video_info, codec="mp4v") as sink:
        for frame_idx, frame in tqdm(enumerate(frame_generator)):
            if frame_idx < len(detection_results) and len(detection_results[frame_idx]["detections"]) > 0:
                detections = from_detection_to_sv(detection_results[frame_idx], color_by=color_by)

                labels = []
                for i in range(len(detections.tracker_id)):
                    label_parts = []
                    if show_attributes:
                        for key, values in detections.data.items():
                            if values is not None and len(values) > i:
                                value = values[i]
                                if isinstance(value, dict):
                                    dict_string = "\n".join(f"{k}:{v}" for k, v in value.items() if (v is not None and v != "none"))
                                    label_parts.append(dict_string)
                                elif value is not None and value != "none":
                                    label_parts.append(f"{key}: {value}")
                    labels.append("\n".join(label_parts))

                annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
                annotated_frame = box_fill_annotator.annotate(scene=annotated_frame, detections=detections)
                annotated_label_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
            else:
                annotated_label_frame = frame.copy()

            annotated_label_frame = cv2.putText(annotated_label_frame, str(frame_idx), org=(10, 20), color=(255, 255, 255), fontFace=1, fontScale=2)
            sink.write_frame(frame=annotated_label_frame)


def reencode_video_w_audio(target_video_path: str, orig_video_path: str, codec="libx264"):
    coded_video_path = str(Path(target_video_path).parent / (Path(target_video_path).stem + "_" + codec + ".mp4"))
    (
        ffmpeg.input(target_video_path).video
        .output(
            ffmpeg.input(orig_video_path).audio,
            coded_video_path,
            vcodec="libx264",
            pix_fmt="yuv420p",
            video_bitrate="12048k",
            acodec="copy",
            **{"profile:v": "high"},
        )
        .run(quiet=True)
    )
    os.remove(target_video_path)
    os.rename(coded_video_path, target_video_path)

## 3a. Visualize the ground-truth annotations

Pick any video id present under the downloaded `videos/test/S1/C6/` folder —
its annotation JSON lives at the matching path under `annotations/`.

In [ ]:
video_id = "S1_C6_F288_V0223"

video_path = DATA_ROOT / "videos" / "test" / SITE_CAMERA / f"{video_id}.mp4"
annot_path = DATA_ROOT / "annotations" / "test" / SITE_CAMERA / f"{video_id}.json"
output_path = f"./{video_id}_gt.mp4"

process_video(str(video_path), str(annot_path), output_path, color_by="track", show_attributes=True)
reencode_video_w_audio(output_path, str(video_path))

## 3b. Visualize a SALMA prediction instead

`process_video` only cares about the JSON schema, not where it comes from — so
running inference with the SALMA submodule (`models/salma`) and pointing
`annot_path` at its output JSON visualizes model predictions in exactly the
same way. Update `salma_pred_path` below to try it.

In [ ]:
salma_pred_path = "/path/to/your/salma_predictions/{}.json".format(video_id)  # <- set this to your SALMA output
output_path = f"./{video_id}_pred.mp4"

process_video(str(video_path), salma_pred_path, output_path, color_by="action", show_attributes=True)
reencode_video_w_audio(output_path, str(video_path))